# Migrate and Verify Elasticsearch Vectors in Qdrant

**Level:** Intermediate · **Stack:** Python, Elasticsearch, Qdrant Migration Tool, Qdrant · **Time:** About 20 minutes, plus downloads

Migrate five documents with known cosine similarities, then verify document counts, original IDs, payloads, stored vectors, rankings, and score conversion. This small fixture checks migration behavior; it is not a recall or performance benchmark.

Use a **Linux x86-64 or ARM64 CPU runtime with Python 3.12 or newer**, at least **4 GB available RAM**, and **5 GB free disk space**. The notebook downloads about 900 MB of official pinned binaries and starts temporary local services. It does not need API keys or Docker. In Google Colab, choose **Runtime → Change runtime type → CPU**, then run cells in order. Google Colab uses x86-64. Its hosted runtime does not provide a Docker daemon, so this notebook runs native binaries instead of the Docker commands in the [tutorial](https://qdrant.tech/documentation/migrate-to-qdrant/from-elasticsearch/).

The databases bind only to loopback and have authentication disabled for this disposable fixture. Do not use these settings for a shared or production deployment. The final cell stops the processes and removes only this notebook's temporary files.

Pinned versions: Elasticsearch **9.5.5**, Qdrant **1.19.2**, Qdrant Migration Tool **0.6.12**, and HTTPX **0.28.1**. Download digests are checked before extraction.


## 1. Install the HTTP Client

The notebook uses the databases' HTTP APIs for setup and verification. The migration tool uses Qdrant's gRPC port.


In [ ]:
%pip -q install httpx==0.28.1

## 2. Prepare an Isolated Runtime

All data, binaries, and logs go into a new temporary directory. The service ports must be unused in this runtime. Elasticsearch refuses to run as root; if the notebook kernel runs as root, only its Elasticsearch subprocess runs under the existing `nobody` account.


In [ ]:
import asyncio
import atexit
import hashlib
import json
import math
import os
import platform
import pwd
import shutil
import signal
import socket
import subprocess
import sys
import tarfile
import tempfile
import time
from datetime import UTC, datetime
from pathlib import Path
from typing import Any

import httpx

if platform.system() != "Linux" or platform.machine() not in {"x86_64", "aarch64"}:
    raise RuntimeError(
        "Use a native Linux x86-64 or ARM64 CPU runtime for these pinned binaries."
    )
if sys.version_info < (3, 12):
    raise RuntimeError("Use Python 3.12 or newer.")

ES_VERSION = "9.5.5"
QDRANT_VERSION = "1.19.2"
MIGRATION_VERSION = "0.6.12"
ES_URL = "http://127.0.0.1:19200"
QDRANT_URL = "http://127.0.0.1:16333"
QDRANT_GRPC_URL = "http://127.0.0.1:16334"
INDEX = COLLECTION = "migration-cosine-check"

for port in (19200, 19300, 16333, 16334):
    with socket.socket() as probe:
        probe.bind(("127.0.0.1", port))

runtime = Path(tempfile.mkdtemp(prefix="qdrant-elasticsearch-migration-"))
runtime.chmod(0o755)
processes: list[subprocess.Popen[bytes]] = []


def stop_services() -> None:
    for process in reversed(processes):
        if process.poll() is None:
            os.killpg(process.pid, signal.SIGTERM)
            try:
                process.wait(timeout=30)
            except subprocess.TimeoutExpired:
                os.killpg(process.pid, signal.SIGKILL)
                process.wait(timeout=10)
    processes.clear()


atexit.register(stop_services)
print(f"Python {platform.python_version()}, {platform.system()} {platform.machine()}")
print("Temporary runtime prepared; all four service ports are available.")

## 3. Download and Verify the Binaries

Elasticsearch includes its own Java runtime. The Elasticsearch SHA-512 digests come from its published [x86-64](https://artifacts.elastic.co/downloads/elasticsearch/elasticsearch-9.5.5-linux-x86_64.tar.gz.sha512) and [ARM64](https://artifacts.elastic.co/downloads/elasticsearch/elasticsearch-9.5.5-linux-aarch64.tar.gz.sha512) checksum files. The Qdrant SHA-256 digest comes from its [GitHub release asset metadata](https://api.github.com/repos/qdrant/qdrant/releases/tags/v1.19.2); the migration digest comes from its [release checksums](https://github.com/qdrant/migration/releases/download/v0.6.12/checksums.txt).


In [ ]:
architecture = platform.machine()
release_arch = "amd64" if architecture == "x86_64" else "arm64"
DIGESTS = {
    "x86_64": {
        "elasticsearch": (
            "36bf5d6897faeda3743841dfaece365330bb41880f907e4c45958964dd2803617d"
            "503cba66e99686124899cff944c983b01a2e08eaebd8f54c26db02bf709ced"
        ),
        "qdrant": "50b253243309ed0ae50a19f678f0d0adcc4f0b567b3bd2c5c0c7e02fa8404bb6",
        "migration": "181aba2003ee4642a9f34a9cd6b39fe6d04f99869c341060adcf86d23ea98d56",
    },
    "aarch64": {
        "elasticsearch": (
            "bbf96d906aa66aa8413f26f848aa4d0c200058a062b87961ceb7e4e0953ec83871"
            "c99620365ff1271e03a5b8a73e0ce44645b415ab509c4cec4327055878e6f9"
        ),
        "qdrant": "6970b93b56fa1203f0cea47d2f330fdb654988fd56617aa77e8478cffd3c0ccb",
        "migration": "ba037845c8493ed6d98064e5858836f1114e10f9e3ffd8a1051fd46d13b08cf2",
    },
}
ARTIFACTS = {
    "elasticsearch": (
        (
            "https://artifacts.elastic.co/downloads/elasticsearch/"
            f"elasticsearch-{ES_VERSION}-linux-{architecture}.tar.gz"
        ),
        "sha512",
        DIGESTS[architecture]["elasticsearch"],
    ),
    "qdrant": (
        (
            f"https://github.com/qdrant/qdrant/releases/download/v{QDRANT_VERSION}/"
            f"qdrant-{architecture}-unknown-linux-musl.tar.gz"
        ),
        "sha256",
        DIGESTS[architecture]["qdrant"],
    ),
    "migration": (
        (
            f"https://github.com/qdrant/migration/releases/download/v{MIGRATION_VERSION}/"
            f"qdrant-migration-linux-{release_arch}.tar.gz"
        ),
        "sha256",
        DIGESTS[architecture]["migration"],
    ),
}


async def download_and_extract(
    client: httpx.AsyncClient, name: str, url: str, algorithm: str, digest: str
) -> None:
    archive = runtime / f"{name}.tar.gz"
    checksum = hashlib.new(algorithm)
    print(f"Downloading {name}...")
    async with client.stream("GET", url) as response:
        response.raise_for_status()
        with archive.open("wb") as output:
            async for chunk in response.aiter_bytes():
                output.write(chunk)
                checksum.update(chunk)
    if checksum.hexdigest() != digest:
        raise ValueError(f"Checksum mismatch: {name}")
    destination = runtime / name
    destination.mkdir()
    with tarfile.open(archive) as package:
        package.extractall(destination, filter="data")
    archive.unlink()
    print(f"Verified and extracted {name}.")


async with httpx.AsyncClient(follow_redirects=True, timeout=300) as downloads:
    await asyncio.gather(
        *(
            download_and_extract(downloads, name, *artifact)
            for name, artifact in ARTIFACTS.items()
        )
    )

es_home = runtime / "elasticsearch" / f"elasticsearch-{ES_VERSION}"
qdrant_binary = runtime / "qdrant" / "qdrant"
migration_binary = runtime / "migration" / "qdrant-migration"
print("All three pinned releases are ready.")

## 4. Start Elasticsearch and Qdrant

Both services run as child processes. Loopback URLs refer to this notebook runtime, not to your laptop. The 512 MB Elasticsearch heap is enough for this fixture. Disabling memory-mapped storage avoids changing the runtime's `vm.max_map_count` setting; it is not a production tuning recommendation.

Use a native Linux runtime with seccomp support. Elasticsearch can fail to start under CPU emulation because its execution sandbox cannot be installed.


In [ ]:
es_config = """cluster.name: migration-notebook
node.name: migration-notebook
discovery.type: single-node
network.host: 127.0.0.1
http.port: 19200
transport.port: 19300
xpack.security.enabled: false
xpack.ml.enabled: false
node.store.allow_mmap: false
"""
(es_home / "config" / "elasticsearch.yml").write_text(es_config)
qdrant_config = runtime / "qdrant.yaml"
qdrant_config.write_text(
    "service:\n  host: 127.0.0.1\n  http_port: 16333\n  grpc_port: 16334\n"
    f"storage:\n  storage_path: {runtime / 'qdrant-data'}\n"
    f"  snapshots_path: {runtime / 'qdrant-snapshots'}\n"
    "telemetry_disabled: true\n"
)

es_uid: int | None = None
es_gid: int | None = None
if os.geteuid() == 0:
    account = pwd.getpwnam("nobody")
    for path in [runtime / "elasticsearch", *es_home.rglob("*"), es_home]:
        os.chown(path, account.pw_uid, account.pw_gid)
    es_uid, es_gid = account.pw_uid, account.pw_gid


def start_service(
    name: str,
    command: list[str],
    cwd: Path,
    env: dict[str, str],
    user_id: int | None = None,
    group_id: int | None = None,
) -> subprocess.Popen[bytes]:
    with (runtime / f"{name}.log").open("wb") as log:
        process = subprocess.Popen(
            command,
            cwd=cwd,
            env=env,
            stdout=log,
            stderr=subprocess.STDOUT,
            start_new_session=True,
            text=False,
            user=user_id,
            group=group_id,
            extra_groups=[] if user_id is not None else None,
        )
    processes.append(process)
    return process


async def wait_ready(name: str, url: str, process: subprocess.Popen[bytes]) -> None:
    deadline = time.monotonic() + 240
    async with httpx.AsyncClient(timeout=5, trust_env=False) as client:
        while time.monotonic() < deadline and process.poll() is None:
            try:
                response = await client.get(url)
                if response.is_success:
                    print(f"{name} is ready.")
                    return
            except httpx.RequestError:
                pass
            await asyncio.sleep(1)
    log = (runtime / f"{name}.log").read_text(errors="replace")
    raise RuntimeError(f"{name} failed to start. Log tail:\n{log[-5000:]}")


es_process = start_service(
    "Elasticsearch",
    [str(es_home / "bin" / "elasticsearch")],
    es_home,
    {
        **os.environ,
        "ES_JAVA_OPTS": "-Xms512m -Xmx512m",
        # Keep JVM container warnings out of the launcher's binary stdout.
        "CLI_JAVA_OPTS": (
            f"{os.environ.get('CLI_JAVA_OPTS', '')} "
            "-Xlog:os+container=off:stdout "
            "-Xlog:os+container=warning:stderr"
        ),
    },
    user_id=es_uid,
    group_id=es_gid,
)
qdrant_process = start_service(
    "Qdrant",
    [str(qdrant_binary), "--config-path", str(qdrant_config)],
    runtime,
    dict(os.environ),
)
try:
    await asyncio.gather(
        wait_ready("Elasticsearch", ES_URL, es_process),
        wait_ready("Qdrant", QDRANT_URL, qdrant_process),
    )
except Exception:
    stop_services()
    raise


async def request(method: str, url: str, **kwargs: Any) -> dict[str, Any]:
    async with httpx.AsyncClient(timeout=60, trust_env=False) as client:
        response = await client.request(method, url, **kwargs)
        response.raise_for_status()
        return response.json()


es_info, qdrant_info = await asyncio.gather(
    request("GET", ES_URL), request("GET", QDRANT_URL)
)
assert es_info["version"]["number"] == ES_VERSION
assert qdrant_info["version"] == QDRANT_VERSION
print(f"Verified Elasticsearch {ES_VERSION} and Qdrant {QDRANT_VERSION}.")

## 5. Create the Source Index and Five Documents

The query vector will be `[1, 0, 0]`. The five unit vectors have cosine similarities `1`, `0.8`, `0`, `-0.6`, and `-1`, so the expected order is unambiguous. A `flat` Elasticsearch index and exact Qdrant search avoid approximate-search differences in this check.

Elasticsearch 9.5 excludes vectors from `_source` by default. The migration tool reads them through the `fields` API. Metadata includes strings, an array, a nested object, an integer, and a boolean.


In [ ]:
QUERY = [1.0, 0.0, 0.0]
VECTORS = {
    "aligned": [1.0, 0.0, 0.0],
    "near": [0.8, 0.6, 0.0],
    "orthogonal": [0.0, 1.0, 0.0],
    "negative": [-0.6, 0.8, 0.0],
    "opposite": [-1.0, 0.0, 0.0],
}
DOCUMENTS = {
    doc_id: {
        "embedding": vector,
        "title": f"Fixture: {doc_id}",
        "category": "cosine-validation",
        "tags": ["fixture", doc_id],
        "metadata": {"ordinal": position, "verified": True},
    }
    for position, (doc_id, vector) in enumerate(VECTORS.items())
}
MAPPING = {
    "settings": {"number_of_shards": 1, "number_of_replicas": 0},
    "mappings": {
        "properties": {
            "embedding": {
                "type": "dense_vector",
                "dims": 3,
                "index": True,
                "similarity": "cosine",
                "index_options": {"type": "flat"},
            },
            "title": {"type": "text"},
            "category": {"type": "keyword"},
            "tags": {"type": "keyword"},
            "metadata": {
                "properties": {
                    "ordinal": {"type": "integer"},
                    "verified": {"type": "boolean"},
                }
            },
        }
    },
}
await request("PUT", f"{ES_URL}/{INDEX}", json=MAPPING)
lines = []
for doc_id, document in DOCUMENTS.items():
    lines.extend((json.dumps({"index": {"_id": doc_id}}), json.dumps(document)))
bulk_result = await request(
    "POST",
    f"{ES_URL}/{INDEX}/_bulk?refresh=true",
    content="\n".join(lines) + "\n",
    headers={"Content-Type": "application/x-ndjson"},
)
assert not bulk_result["errors"], bulk_result
source_count = await request("GET", f"{ES_URL}/{INDEX}/_count")
assert source_count["count"] == 5
print("Created the source index and indexed 5 documents.")

## 6. Run the Migration Tool

The tool creates the destination collection and the named vector `embedding`. The small batch size exercises more than one batch. Original Elasticsearch IDs are stored in payload field `__id__`; arbitrary string IDs become deterministic UUID point IDs.

Migration uses the gRPC endpoint `16334`. Verification uses the HTTP endpoint `16333`. If you repeat the migration, first clean up and rerun the notebook to create a fresh isolated source and destination.


In [ ]:
command = [
    str(migration_binary),
    "elasticsearch",
    "--elasticsearch.url",
    ES_URL,
    "--elasticsearch.index",
    INDEX,
    "--qdrant.url",
    QDRANT_GRPC_URL,
    "--qdrant.collection",
    COLLECTION,
    "--migration.batch-size",
    "2",
]
migration = await asyncio.create_subprocess_exec(
    *command, stdout=asyncio.subprocess.PIPE, stderr=asyncio.subprocess.STDOUT
)
try:
    migration_output, _ = await asyncio.wait_for(migration.communicate(), timeout=180)
except TimeoutError:
    migration.kill()
    await migration.wait()
    raise
print(migration_output.decode())
assert migration.returncode == 0, "Migration failed; inspect the migration output."
print("Migration completed.")

## 7. Compare Counts, Data, Rankings, and Scores

For this **pure, unboosted cosine kNN query**, Elasticsearch returns:

`elasticsearch_score = (1 + cosine_similarity) / 2`

Qdrant's cosine score is the cosine similarity itself. Convert the Elasticsearch score with:

`qdrant_score = 2 * elasticsearch_score - 1`

This conversion does **not** apply to BM25 scores, hybrid fusion, boosts, or arbitrary `script_score` formulas. The [Elasticsearch dense-vector reference](https://www.elastic.co/docs/reference/elasticsearch/mapping-reference/dense-vector) defines the scoring formula.

The verification assertions check every fixture document, not only the first result. They compare all payload fields, both services' stored vectors, original IDs, collection configuration, rank order, and converted scores with absolute tolerance `1e-6`. Qdrant normalizes cosine vectors on upload; this fixture uses unit vectors, so their components can be compared directly.


In [ ]:
es_query = {
    "size": 5,
    "knn": {"field": "embedding", "query_vector": QUERY, "k": 5},
    "fields": ["embedding"],
}
qdrant_query = {
    "query": QUERY,
    "using": "embedding",
    "limit": 5,
    "with_payload": True,
    "with_vector": True,
    "params": {"exact": True},
}
source, target, collection, source_count, target_count = await asyncio.gather(
    request("POST", f"{ES_URL}/{INDEX}/_search", json=es_query),
    request(
        "POST", f"{QDRANT_URL}/collections/{COLLECTION}/points/query", json=qdrant_query
    ),
    request("GET", f"{QDRANT_URL}/collections/{COLLECTION}"),
    request("GET", f"{ES_URL}/{INDEX}/_count"),
    request(
        "POST",
        f"{QDRANT_URL}/collections/{COLLECTION}/points/count",
        json={"exact": True},
    ),
)
hits = source["hits"]["hits"]
points = target["result"]["points"]
assert source_count["count"] == target_count["result"]["count"] == 5
assert len(hits) == len(points) == 5
assert (
    [hit["_id"] for hit in hits]
    == [point["payload"]["__id__"] for point in points]
    == list(VECTORS)
)
assert collection["result"]["config"]["params"]["vectors"]["embedding"] == {
    "size": 3,
    "distance": "Cosine",
}

rows = []
for hit, point in zip(hits, points, strict=True):
    doc_id = hit["_id"]
    expected_payload = {
        "__id__": doc_id,
        **{
            key: value for key, value in DOCUMENTS[doc_id].items() if key != "embedding"
        },
    }
    assert point["payload"] == expected_payload
    assert "embedding" not in hit["_source"]
    for actual, expected in zip(
        point["vector"]["embedding"], VECTORS[doc_id], strict=True
    ):
        assert math.isclose(actual, expected, rel_tol=0, abs_tol=1e-6)
    for actual, expected in zip(
        hit["fields"]["embedding"], VECTORS[doc_id], strict=True
    ):
        assert math.isclose(actual, expected, rel_tol=0, abs_tol=1e-6)
    converted = 2 * hit["_score"] - 1
    assert math.isclose(converted, point["score"], rel_tol=0, abs_tol=1e-6)
    assert math.isclose(point["score"], VECTORS[doc_id][0], rel_tol=0, abs_tol=1e-6)
    rows.append(
        {
            "source_id": doc_id,
            "elasticsearch_score": hit["_score"],
            "converted_cosine": converted,
            "qdrant_score": point["score"],
            "absolute_error": abs(converted - point["score"]),
        }
    )

print(f"{'Document':<12} {'ES score':>10} {'2 * ES - 1':>12} {'Qdrant':>10}")
for row in rows:
    print(
        f"{row['source_id']:<12} {row['elasticsearch_score']:>10.6f} "
        f"{row['converted_cosine']:>12.6f} {row['qdrant_score']:>10.6f}"
    )
print("PASS: 5 source documents = 5 destination points.")
print("PASS: original IDs, payloads, unit vectors, configuration, and ranking match.")
print("PASS: all converted scores match within absolute tolerance 1e-6.")
validation = {
    "tested_at": datetime.now(UTC).isoformat(),
    "python_version": platform.python_version(),
    "platform": f"{platform.system()} {platform.machine()}",
    "elasticsearch_version": ES_VERSION,
    "qdrant_version": QDRANT_VERSION,
    "migration_version": MIGRATION_VERSION,
    "source_count": 5,
    "destination_count": 5,
    "batch_size": 2,
    "score_absolute_tolerance": 1e-6,
    "max_absolute_score_error": max(row["absolute_error"] for row in rows),
    "rows": rows,
}
print(json.dumps(validation, indent=2))

Expected ranking and scores:

| Document | Elasticsearch | Converted cosine | Qdrant |
|---|---:|---:|---:|
| aligned | 1.0 | 1.0 | 1.0 |
| near | 0.9 | 0.8 | 0.8 |
| orthogonal | 0.5 | 0.0 | 0.0 |
| negative | 0.2 | -0.6 | -0.6 |
| opposite | 0.0 | -1.0 | -1.0 |

Small floating-point differences are expected. This fixture confirms these pinned versions transfer the illustrated fields and preserve exact cosine ranking. It does not establish parity for production filters, approximate search, other vector types, hybrid scoring, authentication, or concurrent writes. For a real migration, stop writes or plan a separate synchronization step, validate representative queries and payloads, and rebuild payload indexes and application query logic as needed.


## 8. Clean Up

Run this cell even if an earlier migration or verification cell fails. It stops only the database child processes created by this notebook, then deletes their temporary directory. To repeat the experiment, rerun from step 2.


In [ ]:
stop_services()
shutil.rmtree(runtime)
atexit.unregister(stop_services)
print("Stopped the notebook's database processes and removed its temporary files.")

## Apply the Check to Your Data

Keep the source and destination comparison, but replace the fixture with representative documents and queries. Compare normalized vectors for cosine collections. Recalibrate score thresholds using `2 * score - 1` only for pure cosine kNN scores; evaluate hybrid retrieval separately.

For Docker networking by platform, migration flags, supported mappings, and limitations, use the [Elasticsearch migration reference](https://qdrant.tech/documentation/migrate-to-qdrant/from-elasticsearch/). For broader cutover checks, use the [Migration Verification Guide](https://qdrant.tech/documentation/migration-guidance/).
